# Hypothesis 1: ingredient network core

Network can contain core from most common ingredients



In [ ]:
import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import scipy
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 80)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RANDOM_SEED = 42
GRAPH_PATH = Path("ingredient_network.graphml")
WEIGHT = "recipe_count"

TOP_N = 25
TOP_EXTREME_N = 25
CORE_VIS_N = 60

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "font.size": 10,
})


In [ ]:
G = nx.read_graphml(GRAPH_PATH)
G = nx.Graph(G)

for node, attrs in G.nodes(data=True):
    attrs[WEIGHT] = int(attrs.get(WEIGHT, 0) or 0)
    for attr_name in ["cuisines", "courses", "time_groups"]:
        value = attrs.get(attr_name)
        if isinstance(value, str):
            try:
                attrs[attr_name] = json.loads(value)
            except json.JSONDecodeError:
                pass

for u, v, attrs in G.edges(data=True):
    attrs[WEIGHT] = int(attrs.get(WEIGHT, 1) or 1)
    for attr_name in ["cuisines", "courses", "time_groups"]:
        value = attrs.get(attr_name)
        if isinstance(value, str):
            try:
                attrs[attr_name] = json.loads(value)
            except json.JSONDecodeError:
                pass

self_loops = list(nx.selfloop_edges(G))
if self_loops:
    G.remove_edges_from(self_loops)

print(f"Nodes: {G.number_of_nodes():,}")
print(f"Edges: {G.number_of_edges():,}")
print(f"Directed: {G.is_directed()}")
print(f"Density: {nx.density(G):.6f}")
print(f"Connected components: {nx.number_connected_components(G):,}")
print(f"Isolates: {nx.number_of_isolates(G):,}")


In [ ]:
from ipysigma import Sigma
Sigma(G, node_size=G.degree)

In [ ]:
for node, attrs in G.nodes(data=True):
    cuisines = attrs.get("cuisines", {})
    if cuisines:
        attrs["main_cuisine"] = max(cuisines, key=cuisines.get)
    else:
        attrs["main_cuisine"] = "unknown"

Sigma(
    G,
    node_size=G.degree,
    node_color="main_cuisine",
    node_color_palette="Tableau10",
)


## 2. Metrics for the hypotheses

We use:

- `recipe_count` for the frequency core;
- `degree` for connectivity extremeness;
- `PageRank` and `eigenvector centrality` for centrality;
- `core_number` and the maximum `k-core` for the structural core.


In [ ]:
recipe_count = {node: attrs.get(WEIGHT, 0) for node, attrs in G.nodes(data=True)}
degree = dict(G.degree())
weighted_degree = dict(G.degree(weight=WEIGHT))
core_number = nx.core_number(G)
pagerank = nx.pagerank(G, alpha=0.85, weight=WEIGHT, max_iter=500, tol=1e-10)
eigenvector = nx.eigenvector_centrality(G, weight=WEIGHT, max_iter=2_000, tol=1e-7)

metrics = pd.DataFrame({
    "ingredient": list(G.nodes),
    "recipe_count": [recipe_count[n] for n in G.nodes],
    "degree": [degree[n] for n in G.nodes],
    "core_number": [core_number[n] for n in G.nodes],
    "pagerank": [pagerank[n] for n in G.nodes],
    "eigenvector": [eigenvector[n] for n in G.nodes],
}).set_index("ingredient")

metrics["recipe_share_of_max"] = metrics["recipe_count"] / metrics["recipe_count"].max()
metrics["degree_share_of_max"] = metrics["degree"] / metrics["degree"].max()

for col in ["recipe_count", "degree", "core_number", "pagerank", "eigenvector"]:
    metrics[f"{col}_pct_rank"] = metrics[col].rank(pct=True)

max_core_number = int(metrics["core_number"].max())
metrics["in_max_core"] = metrics["core_number"].eq(max_core_number)
metrics["frequency_extreme_99p"] = metrics["recipe_count"].ge(metrics["recipe_count"].quantile(0.99))
metrics["degree_extreme_99p"] = metrics["degree"].ge(metrics["degree"].quantile(0.99))


summary_cols = [
    "recipe_count", "recipe_share_of_max", "degree", "degree_share_of_max",
     "core_number",
    "pagerank", "eigenvector", "in_max_core",
]

metrics[summary_cols].sort_values("recipe_count", ascending=False).head(TOP_N)


## 3. Get Hubs covering degree share

In [ ]:
def get_hubs_covering_degree_share(metrics, n=0.3, degree_col="degree"):
    degrees = metrics[degree_col].sort_values(ascending=False)
    degrees = degrees[degrees > 0]

    total_degree = degrees.sum()
    target_degree = total_degree * n

    cumulative_degree = degrees.cumsum()

    hubs = cumulative_degree[cumulative_degree <= target_degree].index.tolist()

    if len(hubs) < len(degrees):
        next_node = cumulative_degree.index[len(hubs)]
        hubs.append(next_node)

    hubs_metrics = metrics.loc[hubs].copy()
    hubs_metrics["degree_share"] = hubs_metrics[degree_col] / total_degree
    hubs_metrics["cumulative_degree_share"] = hubs_metrics[degree_col].cumsum() / total_degree

    summary = {
        "target_share": n,
        "actual_share": hubs_metrics[degree_col].sum() / total_degree,
        "hub_count": len(hubs),
        "total_nodes": len(degrees),
        "hub_node_share": round(len(hubs) / len(degrees) * 100, 2),
        "total_degree": total_degree,
        "hub_degree_sum": hubs_metrics[degree_col].sum(),
    }

    return hubs_metrics, summary, hubs


In [ ]:
hubs_20, summary_20, hubs20_nodes = get_hubs_covering_degree_share(metrics, n=0.20)
hubs_25, summary_25, hubs25_nodes = get_hubs_covering_degree_share(metrics, n=0.25)
hubs_30, summary_30, hubs30_nodes = get_hubs_covering_degree_share(metrics, n=0.30)
hubs_40, summary_40, hubs40_nodes = get_hubs_covering_degree_share(metrics, n=0.40)
hubs_50, summary_50, hubs50_nodes = get_hubs_covering_degree_share(metrics, n=0.50)

summary_20, summary_25, summary_30, summary_40, summary_50


In [ ]:
true_hubs = hubs20_nodes

## 3. Frequency core


In [ ]:
frequency_top = metrics[summary_cols].sort_values("recipe_count", ascending=False).head(TOP_N)
display(frequency_top)

fig, ax = plt.subplots(figsize=(11, 5))
values = metrics["recipe_count"].sort_values(ascending=False).reset_index(drop=True)
ax.plot(values.index + 1, values.values, linewidth=2)
ax.scatter(np.arange(1, min(TOP_N, len(values)) + 1), values.iloc[:TOP_N], s=30)
ax.set_title("Ingredient recipe_count rank curve")
ax.set_xlabel("Ingredient rank by recipe_count")
ax.set_ylabel("recipe_count")
ax.grid(True, alpha=0.25)
plt.show()


## 4. Structural core


In [ ]:
print("Top ingredients by degree:")
display(metrics[summary_cols].sort_values("degree", ascending=False).head(TOP_N))

print("Top ingredients by PageRank:")
display(metrics[summary_cols].sort_values("pagerank", ascending=False).head(TOP_N))

print("Top ingredients inside maximum k-core:")
display(
    metrics.loc[metrics["in_max_core"], summary_cols]
    .sort_values(["recipe_count", "degree"], ascending=False)
    .head(TOP_N)
)


In [ ]:
from itertools import combinations

TOP_N = 60

corr_cols = ["recipe_count", "degree", "core_number", "pagerank", "eigenvector"]
print("Spearman correlation between frequency and structural metrics:")
display(metrics[corr_cols].corr(method="spearman"))

sets = {
    "top_recipe_count": set(metrics.nlargest(TOP_N, "recipe_count").index),
    "top_degree": set(metrics.nlargest(TOP_N, "degree").index),
    "top_pagerank": set(metrics.nlargest(TOP_N, "pagerank").index),
    "top_eigenvector": set(metrics.nlargest(TOP_N, "eigenvector").index),
    "max_core": set(metrics.index[metrics["in_max_core"]]),
}

overlap_rows = []
a_name, a_nodes = "selected_hubs", set(true_hubs)
for b_name, b_nodes in sets.items():
    if a_name >= b_name:
        continue
    overlap_rows.append({
        "set_a": a_name,
        "set_b": b_name,
        "overlap": len(a_nodes & b_nodes),
        "share_of_a": len(a_nodes & b_nodes) / len(a_nodes),
        "share_of_b": len(a_nodes & b_nodes) / len(b_nodes),
        "difference": a_nodes ^ b_nodes,
    })

display(pd.DataFrame(overlap_rows).sort_values("overlap", ascending=False))


## 5. Degree distribution

We inspect the distribution of ordinary degree and weighted degree. If the network has hubs or a core, the distribution should be strongly right-skewed.


In [ ]:
axes = plt.gca()

axes.hist(metrics["degree"], bins=50, color="#3b82f6", edgecolor="white")
axes.axvline(metrics["degree"].median(), color="#111827", linestyle="--", linewidth=1.5, label="median")
axes.axvline(metrics["degree"].quantile(0.95), color="#ef4444", linestyle="--", linewidth=1.5, label="95th percentile")
axes.set_title("Degree distribution")
axes.set_xlabel("degree")
axes.set_ylabel("number of ingredients")
axes.legend()


plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

degrees = metrics.loc[metrics["degree"] > 0, "degree"].to_numpy()
degrees = np.sort(degrees)

unique_degrees = np.unique(degrees)
ccdf = np.array([(degrees >= k).mean() for k in unique_degrees])

plt.figure(figsize=(8, 5))
plt.loglog(unique_degrees, ccdf, marker=".", linestyle="none")
plt.xlabel("degree k")
plt.ylabel("P(K >= k)")
plt.title("Degree distribution CCDF")
plt.tight_layout()
plt.show()

## 6. Extreme-degree nodes and their links

We take the top-N ingredients by degree and inspect their induced subgraph: number of links, density, and the strongest co-occurrence pairs inside this extreme set.


In [ ]:
H_extreme = G.subgraph(true_hubs).copy()

possible_edges = H_extreme.number_of_nodes() * (H_extreme.number_of_nodes() - 1) / 2
extreme_density = nx.density(H_extreme)
extreme_weights = [attrs.get(WEIGHT, 1) for _, _, attrs in H_extreme.edges(data=True)]

print(f"Extreme-degree nodes: {H_extreme.number_of_nodes():,}")
print(f"Edges among them: {H_extreme.number_of_edges():,} / {possible_edges:,.0f}")
print(f"Density among extreme-degree nodes: {extreme_density:.4f}")
print(f"Full graph density: {nx.density(G):.6f}")
print(f"Median edge recipe_count inside extreme subgraph: {np.median(extreme_weights):.2f}")
print(f"Max edge recipe_count inside extreme subgraph: {np.max(extreme_weights):.2f}")

print("Extreme-degree nodes:")
display(metrics.loc[true_hubs, summary_cols].sort_values("degree", ascending=False))

edge_rows = []
for u, v, attrs in H_extreme.edges(data=True):
    edge_rows.append({
        "ingredient_a": u,
        "ingredient_b": v,
        "edge_recipe_count": attrs.get(WEIGHT, 1),
        "a_degree": metrics.loc[u, "degree"],
        "b_degree": metrics.loc[v, "degree"],
        "a_recipe_count": metrics.loc[u, "recipe_count"],
        "b_recipe_count": metrics.loc[v, "recipe_count"],
    })

edge_table_extreme = pd.DataFrame(edge_rows).sort_values("edge_recipe_count", ascending=False)
print("Strongest links among extreme-degree ingredients:")
display(edge_table_extreme.head(40))


In [ ]:
low_degree_nodes = metrics.sort_values(["degree", "recipe_count"], ascending=[True, False]).head(25).index.tolist()
print("Low-degree extreme nodes, for contrast:")
display(metrics.loc[low_degree_nodes, summary_cols].sort_values(["degree", "recipe_count"], ascending=[True, False]))


## 7. Core visualizations

In [ ]:
hub_graph = G.subgraph(true_hubs).copy()

Sigma(hub_graph, node_size=hub_graph.degree, )

# Hypothesis 2: Cusine clustering

If we remove ingredients from hypothesis 1, network would better cluser around cusine

In [ ]:
graph_without_hubs = G.copy()
graph_without_hubs.remove_nodes_from(true_hubs)

for node, attrs in graph_without_hubs.nodes(data=True):
    cuisines = attrs.get("cuisines", {})
    if cuisines:
        attrs["main_cuisine"] = max(cuisines, key=cuisines.get)
    else:
        attrs["main_cuisine"] = "unknown"

Sigma(
    graph_without_hubs,
    node_size=graph_without_hubs.degree,
    node_color="main_cuisine",
    node_color_palette="Tableau10",
)


In [ ]:
import networkx as nx
import pandas as pd
from collections import defaultdict

WEIGHT = "recipe_count"


In [ ]:
def add_main_cuisine(graph):
    graph = graph.copy()

    for node, attrs in graph.nodes(data=True):
        cuisines = attrs.get("cuisines", {})

        if isinstance(cuisines, str):
            import json
            cuisines = json.loads(cuisines)

        if cuisines:
            attrs["main_cuisine"] = max(cuisines, key=cuisines.get)
        else:
            attrs["main_cuisine"] = "unknown"

    return graph


In [ ]:
G_labeled = add_main_cuisine(G)

core_nodes = true_hubs

G_no_core = G_labeled.copy()
G_no_core.remove_nodes_from(core_nodes)
G_no_core.remove_nodes_from(list(nx.isolates(G_no_core)))


In [ ]:
def same_cuisine_edge_share(graph, cuisine_attr="main_cuisine"):
    same = 0
    total = 0

    for u, v in graph.edges():
        cu = graph.nodes[u].get(cuisine_attr)
        cv = graph.nodes[v].get(cuisine_attr)

        if cu == "unknown" or cv == "unknown":
            continue

        total += 1
        if cu == cv:
            same += 1

    return same / total if total else None


def weighted_same_cuisine_edge_share(graph, cuisine_attr="main_cuisine", weight=WEIGHT):
    same_weight = 0
    total_weight = 0

    for u, v, attrs in graph.edges(data=True):
        cu = graph.nodes[u].get(cuisine_attr)
        cv = graph.nodes[v].get(cuisine_attr)

        if cu == "unknown" or cv == "unknown":
            continue

        w = attrs.get(weight, 1)
        total_weight += w

        if cu == cv:
            same_weight += w

    return same_weight / total_weight if total_weight else None


def cuisine_modularity(graph, cuisine_attr="main_cuisine", weight=WEIGHT):
    groups = defaultdict(set)

    for node, attrs in graph.nodes(data=True):
        cuisine = attrs.get(cuisine_attr, "unknown")
        if cuisine != "unknown":
            groups[cuisine].add(node)

    communities = [nodes for nodes in groups.values() if len(nodes) > 0]

    return nx.algorithms.community.quality.modularity(
        graph,
        communities,
        weight=weight
    )


def graph_cuisine_metrics(graph, name):
    return {
        "graph": name,
        "nodes": graph.number_of_nodes(),
        "edges": graph.number_of_edges(),
        "density": nx.density(graph),
        "components": nx.number_connected_components(graph),
        "largest_component_share": len(max(nx.connected_components(graph), key=len)) / graph.number_of_nodes(),
        "cuisine_assortativity": nx.attribute_assortativity_coefficient(graph, "main_cuisine"),
        "same_cuisine_edge_share": same_cuisine_edge_share(graph),
        "weighted_same_cuisine_edge_share": weighted_same_cuisine_edge_share(graph),
        "cuisine_modularity": cuisine_modularity(graph),
        "avg_clustering": nx.average_clustering(graph, weight=WEIGHT),
    }


In [ ]:
import random
import numpy as np

def random_removal_baseline(graph, remove_count, n_iter=200, seed=42):
    rng = random.Random(seed)
    nodes = list(graph.nodes)

    rows = []

    for i in range(n_iter):
        removed_nodes = rng.sample(nodes, remove_count)

        H = graph.copy()
        H.remove_nodes_from(removed_nodes)
        H.remove_nodes_from(list(nx.isolates(H)))

        if H.number_of_nodes() == 0 or H.number_of_edges() == 0:
            continue

        row = graph_cuisine_metrics(H, f"random_{i}")
        rows.append(row)

    return pd.DataFrame(rows)


In [ ]:
random_baseline = random_removal_baseline(
    G_labeled,
    remove_count=len(core_nodes),
    n_iter=200,
    seed=42
)

before_row = graph_cuisine_metrics(G_labeled, "before")
after_row = graph_cuisine_metrics(G_no_core, "after_core_removed")

random_mean_row = random_baseline.mean(numeric_only=True).to_dict()
random_mean_row["graph"] = "random_removal_mean"

random_median_row = random_baseline.median(numeric_only=True).to_dict()
random_median_row["graph"] = "random_removal_median"

random_p95_row = random_baseline.quantile(0.95, numeric_only=True).to_dict()
random_p95_row["graph"] = "random_removal_95pct"


In [ ]:
comparison = pd.DataFrame([
    before_row,
    random_mean_row,
    after_row,
])

comparison